In [1]:
import json
import os
import pandas as pd
from Bio import SeqIO

In [2]:
root = "D:/Virology/RNA_viruses/Astrovirus/Astroviridae_analysis/data/"
bastro_capsid = f"{root}bastrovirus_dataset/cds_assigned/bastro_capsid_reduced.fasta"
astro_capsid_nt_all = f"{root}orf_res_amb_all/Astroviridae_complete_2.fna"
clustering_res = f"{root}orf1b_83nt_for_prediction/Astroviridae_complete_1B_nt83_aa0.7_HPC.tsv"


astro_bastro_capsid_path = f"{root}capsid_prediction/Astro_bastro_capsid.fasta"
alphafold_batches = f"{root}capsid_prediction/alphafold3_batches"

In [4]:
clustering_df = pd.read_csv(clustering_res, sep="\t", header=None)
centers = clustering_df[0].unique()

records_astro = SeqIO.parse(astro_capsid_nt_all, "fasta")
records_astro_bastro = []
for rec in records_astro:

    if rec.id in centers:
        rec.seq = rec.seq.translate(to_stop=True)
        records_astro_bastro.append(rec)
print(f"{len(records_astro_bastro)} astrovirus sequences were added")

records_bastro =  SeqIO.parse(bastro_capsid, "fasta")
for rec in records_bastro:
    records_astro_bastro.append(rec)

SeqIO.write(records_astro_bastro, astro_bastro_capsid_path, "fasta")

119 astrovirus sequences were added


C:\Python310\lib\site-packages\Bio\Seq.py:2879: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


137

Create jobs for alphafold3 server

In [36]:
max_length = 4999
max_jobs = 30
list_dicts = []

list_seqs = []
cum_length = 0
num_jobs = 1
for rec in records_astro_bastro:
    if cum_length + len(rec.seq) < max_length:
        cum_length += len(rec.seq)
        list_seqs.append(
                {
            "proteinChain": {
            "sequence": str(rec.seq),
            "count": 1
          } })
    else:
        dict_job = {
            "name": "Job " + str(num_jobs),
            "modelSeeds": [],
            "sequences": list_seqs,
            "dialect": "alphafoldserver",
            "version": 1}
        list_dicts.append(dict_job)
        cum_length = 0
        list_seqs = []
        num_jobs+=1

print(f"Built {len(list_dicts)} job(s)")

with open(f"{root}capsid_prediction/alphafold3_jobs.json", "w") as f:
    json.dump(list_dicts, f, indent=2)

Built 19 job(s)


In [31]:
list_dicts

[{'name': 'Job 1',
  'modelSeeds': '[]',
  'sequences': [{'proteinChain': {'sequence': 'MNIGSRSTSVWLLKSWPGTFAVRGARSPSYSAIGCLTRFGGEDQSSAMAGNAQAAAKAKATIKEVAKEIVKEAKNTNQRQNRPSKRWYKKGRHMPKTNNKKANNQIKQTVEKDVKQKLKKEGLEGPRSRFSVRVSATIGKIGPNREQGPELQIATFMHPSLMKEPNDGTNFGPLQAAAAQWGLWRLSSLEIRFTPLVGSSAVTGSVYRVSLNLTQSPGNASWGGLGARKHRDIPVGKSVSWKLQRGDLAGPRQTWWMTDTNEEGGQSCGPMVEVHGLGKTSSTYQNQDWTGDLFIVEVDGRWEFTNYNAKPALGTLDRKSEDYATDSTGPGIKVGDDGVMKMTLPQASTLARFMSDACERSAINEGSVGETIWQIVDEGAGLASSVAPTPFSWLIKGAWWFVKKLAGRAGENTDEEYVVYASLADAQNNKPVIANRFTHVKKATTLTVTQVNSPNVGPGATLPAYHTESPYPLIPAEVPPPAGSKMLLTCLASPIKINFDNARESVFKATARVGADDYLFSIKLGRNKAFATSGYLLREPVAMTIEGQRITGMYDPFWMKSTGMSVHYRNTDTVIGQVIAYASDNWGTSNSPLTCAYWLVRVTNTLAATEPVNFYNYAPGAYTSGGTTLNPPELYTLNKAGVPTMNRSGGSGHTVCRVLIPRIESQTLMLVYSIGLNTMQSNGLTNTQPITGASNDVNLNVNHSSDFGFWPHALTPIGLTEDRLNLWFTVPPDRPSDESMVDDIINVIQRRFRLRPVETSESEDSSGDDLSESEGEHQEGCHTPPKSNKMKFA',
     'count': 1}},
   {'proteinChain': {'sequence': 'MADRAVAPREKVTKKVTKVVTVKKKHPKKKPKQKVSKPQKLPMKAERKLEKEVKGLKKRIAGP

In [32]:
json_str = json.dumps(list_dicts)

In [34]:
with open(f"{root}capsid_prediction/alphafold3_jobs.json", "w") as file:
    file.write(json_str)
file.close()

In [6]:
len(records_astro_bastro)

137

In [7]:
max_length = 4999
max_jobs = 30
list_dicts = []


batch = 0
num_jobs = 0
for rec in records_astro_bastro:
    if num_jobs >= 30:
        print(len(list_dicts))
        with open(f"{root}capsid_prediction/alphafold3_separate_jobs_{batch}.json", "w") as f:
            json.dump(list_dicts, f, indent=2)
        list_dicts = []
        num_jobs = 0
        batch+=1
    #if cum_length + len(rec.seq) < max_length:
    chain = {
            "proteinChain": {
            "sequence": str(rec.seq),
            "count": 1
          } }
    dict_job = {
            "name":  "capsid " + rec.id.split("/")[0].split("|")[0],
            "modelSeeds": [],
            "sequences": [chain],
            "dialect": "alphafoldserver",
            "version": 1}
    list_dicts.append(dict_job)
    num_jobs +=1

if len(list_dicts) > 1:
    with open(f"{root}capsid_prediction/alphafold3_separate_jobs_{batch}.json", "w") as f:
        json.dump(list_dicts, f, indent=2)

print(f"Built {batch} batches")



30
30
30
30
Built 5 batches
